# RAG Indexing: Chunking Strategies

## 1. Overview
Chunking directly governs the precision-recall trade-off of any retrieval pipeline:
- **Too small**: The retrieved passage lacks the context needed to answer the question, or splits an answer across boundaries.
- **Too large**: The passage contains too much irrelevant text, diluting the embedding vector and wasting LLM context window tokens.

In this notebook, we compare four major chunking strategies on the Wikipedia passages prepared in **`6.5-Storage-Vector-Databases.ipynb`**:
1. **Fixed-size Chunking** (`CharacterTextSplitter`)
2. **Recursive Character Chunking** (`RecursiveCharacterTextSplitter`)
3. **Sentence-aware Chunking** (`NLTKTextSplitter`)
4. **Semantic Chunking** (`SemanticChunker`)

We inspect chunk size distributions and quantitatively benchmark retrieval hit rate across strategies using temporary Qdrant collections.


In [ ]:
%pip install -q langchain langchain-qdrant langchain-huggingface langchain-experimental sentence-transformers qdrant-client nltk matplotlib


In [ ]:
import statistics
from statistics import mean

import matplotlib.pyplot as plt
import nltk
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import (
    CharacterTextSplitter,
    NLTKTextSplitter,
    RecursiveCharacterTextSplitter,
)
from qdrant_client import QdrantClient, models

import rag_common as rc

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

## 1. Load Configuration & Corpus Passages

We load the HotpotQA corpus cached by **`6.5-Storage-Vector-Databases.ipynb`** (or fetch it directly if running standalone).
Each passage is a full Wikipedia paragraph (~200–500 tokens).


In [ ]:
cfg = rc.load_config()
rc.check_services(cfg, llm=False)
QDRANT_URL = cfg["qdrant"]["url"]
EMBED_MODEL = cfg["embeddings"]["model"]

qa_pairs, corpus = rc.load_hotpotqa(cfg)   # cached by 6.5-Storage-Vector-Databases.ipynb

# Chunking is compared on a fixed sample of passages so every strategy sees identical input
raw_docs = [{"text": p["text"], "title": p["title"]} for p in corpus[:100]]

print(f"Loaded {len(qa_pairs)} QA pairs and {len(corpus)} passages; chunking {len(raw_docs)} of them.")
print(f"\nSample passage ({len(raw_docs[0]['text'].split())} words):\n{raw_docs[0]['text'][:300]}...")

---
## 2. Implementing Chunking Strategies

We split the exact same source documents four different ways and inspect the resulting chunks.


### 2a. Fixed-size Chunking

`CharacterTextSplitter` splits on a separator and enforces a hard character limit with overlap.
Simple and fast, but may cut sentences mid-thought.


In [ ]:
fixed_splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
)

fixed_chunks = fixed_splitter.create_documents(
    texts=[d["text"] for d in raw_docs],
    metadatas=[{"title": d["title"]} for d in raw_docs],
)

sizes = [len(c.page_content) for c in fixed_chunks]
print(f"Fixed-size  →  {len(fixed_chunks)} chunks")
print(f"  Avg chars: {mean(sizes):.0f}  |  Min: {min(sizes)}  |  Max: {max(sizes)}")


### 2b. Recursive Character Chunking

`RecursiveCharacterTextSplitter` tries a hierarchy of separators in order:
`\n\n` → `\n` → `. ` → ` ` → `""`

It recurses until chunks are under `chunk_size`, preserving paragraphs and sentences where possible.


In [ ]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

recursive_chunks = recursive_splitter.create_documents(
    texts=[d["text"] for d in raw_docs],
    metadatas=[{"title": d["title"]} for d in raw_docs],
)

sizes = [len(c.page_content) for c in recursive_chunks]
print(f"Recursive   →  {len(recursive_chunks)} chunks")
print(f"  Avg chars: {mean(sizes):.0f}  |  Min: {min(sizes)}  |  Max: {max(sizes)}")


### 2c. Sentence-aware Chunking

`NLTKTextSplitter` uses the NLTK Punkt tokenizer to detect complete sentence boundaries before merging them into chunks.
Guarantees that no sentence is ever broken across chunk boundaries.


In [ ]:
sentence_splitter = NLTKTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

sentence_chunks = sentence_splitter.create_documents(
    texts=[d["text"] for d in raw_docs],
    metadatas=[{"title": d["title"]} for d in raw_docs],
)

sizes = [len(c.page_content) for c in sentence_chunks]
print(f"Sentence    →  {len(sentence_chunks)} chunks")
print(f"  Avg chars: {mean(sizes):.0f}  |  Min: {min(sizes)}  |  Max: {max(sizes)}")


### 2d. Semantic Chunking

`SemanticChunker` embeds individual sentences, then merges adjacent sentences as long as their
embedding cosine distance stays below a threshold percentile. A new chunk is created whenever topic distance spikes.


In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=85,
)

# Run on the 100 sample documents
semantic_chunks = semantic_splitter.create_documents(
    texts=[d["text"] for d in raw_docs],
    metadatas=[{"title": d["title"]} for d in raw_docs],
)

sizes = [len(c.page_content) for c in semantic_chunks]
print(f"Semantic    →  {len(semantic_chunks)} chunks  (from {len(raw_docs)} passages)")
print(f"  Avg chars: {mean(sizes):.0f}  |  Min: {min(sizes)}  |  Max: {max(sizes)}")
print(f"\nExample semantic chunk:\n{semantic_chunks[0].page_content!r}")


---
## 3. Chunk Size Distribution Analysis

Comparing character count distributions highlights how each splitter handles length and variance.


In [ ]:
strategies_chunks = {
    "Fixed-size":  fixed_chunks,
    "Recursive":   recursive_chunks,
    "Sentence":    sentence_chunks,
    "Semantic":    semantic_chunks,
}

print(f"{'Strategy':<14}{'Count':>7}{'Mean':>7}{'Median':>8}{'StdDev':>8}{'Min':>6}{'Max':>6}")
print("-" * 56)
for name, ch in strategies_chunks.items():
    s = [len(c.page_content) for c in ch]
    print(f"{name:<14}{len(s):>7}{mean(s):>7.0f}{statistics.median(s):>8.0f}{statistics.stdev(s):>8.0f}{min(s):>6}{max(s):>6}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: box plot
all_sizes = {name: [len(c.page_content) for c in ch] for name, ch in strategies_chunks.items()}
axes[0].boxplot(all_sizes.values(), tick_labels=all_sizes.keys(), patch_artist=True)
axes[0].set_ylabel("Chunk size (characters)")
axes[0].set_title("Chunk Size Distribution by Strategy")
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

# Right: histograms
for name, ch in strategies_chunks.items():
    axes[1].hist([len(c.page_content) for c in ch], bins=25, alpha=0.5, label=name)
axes[1].set_xlabel("Chunk size (characters)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Chunk Size Histograms")
axes[1].legend()
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


---
## 4. Quantitative Retrieval Evaluation in Qdrant

Index each strategy's chunks into a temporary Qdrant collection and measure retrieval hit rate:
does the gold answer string appear inside the top-k retrieved chunks for the question?


In [ ]:
client = QdrantClient(url=QDRANT_URL)
vector_size = len(embeddings.embed_query("test"))

def build_collection(name, chunks):
    if client.collection_exists(name):
        client.delete_collection(name)
    client.create_collection(
        collection_name=name,
        vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE),
    )
    store = QdrantVectorStore(client=client, collection_name=name, embedding=embeddings)
    store.add_documents(chunks)
    return store

# Evaluate on questions corresponding to the sampled documents
sample_titles = set(d["title"] for d in raw_docs)
eval_pairs = [qa for qa in qa_pairs if any(t in sample_titles for t in qa.get("supporting_titles", []))]
if not eval_pairs:
    eval_pairs = qa_pairs[:50]

print(f"Evaluating across {len(eval_pairs)} questions...")

eval_chunks = {
    "fixed":     fixed_chunks,
    "recursive": recursive_chunks,
    "sentence":  sentence_chunks,
    "semantic":  semantic_chunks,
}

stores = {}
for strategy_name, chunks in eval_chunks.items():
    col = f"chunk_eval_{strategy_name}"
    stores[strategy_name] = build_collection(col, chunks)
    print(f"Indexed {len(chunks):>4} chunks → '{col}'")


In [ ]:
retrieval_hit = rc.retrieval_hit   # shared metric: does the answer text appear in the retrieved context?

K = 3
print(f"Retrieval hit rate  (top-{K}, {len(eval_pairs)} queries)\n")
print(f"{'Strategy':<12}{'Hit Rate':>12}{'Chunks / Doc':>16}")
print("-" * 40)

retrieval_results = {}
for strategy_name, store in stores.items():
    retriever = store.as_retriever(search_kwargs={"k": K})
    hits = []
    for pair in eval_pairs:
        docs = retriever.invoke(pair["question"])
        hits.append(retrieval_hit(docs, pair["answer"]))
    # Chunks per document correctly computed relative to raw source passages
    chunks_per_doc = len(eval_chunks[strategy_name]) / len(raw_docs)
    hit_rate = mean(hits) * 100
    retrieval_results[strategy_name] = {"hit_rate": hit_rate, "chunks_per_doc": chunks_per_doc}
    print(f"{strategy_name:<12}{hit_rate:>11.1f}%{chunks_per_doc:>16.1f}")


In [ ]:
for name, r in retrieval_results.items():
    rc.record_result(f"Chunking: {name}", "chunking*", hit=r["hit_rate"] / 100, n=len(eval_pairs))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

strategy_labels = list(retrieval_results.keys())
hit_rates       = [retrieval_results[s]["hit_rate"] for s in strategy_labels]
chunks_per_doc  = [retrieval_results[s]["chunks_per_doc"] for s in strategy_labels]
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
x = range(len(strategy_labels))

bars = axes[0].bar(x, hit_rates, color=colors, alpha=0.8, edgecolor="white", linewidth=1.2)
axes[0].bar_label(bars, fmt="%.1f%%", padding=3, fontsize=10, fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels([s.capitalize() for s in strategy_labels])
axes[0].set_ylabel("Hit Rate (%)")
axes[0].set_ylim(0, min(max(hit_rates) * 1.3, 100))
axes[0].set_title(f"Retrieval Hit Rate (top-{K})")
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

bars2 = axes[1].bar(x, chunks_per_doc, color=colors, alpha=0.8, edgecolor="white", linewidth=1.2)
axes[1].bar_label(bars2, fmt="%.1f", padding=3, fontsize=10, fontweight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels([s.capitalize() for s in strategy_labels])
axes[1].set_ylabel("Avg Chunks per Document")
axes[1].set_title("Index Density (Chunks / Doc)")
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


---
## 5. Effect of Chunk Size (Recursive Splitter Sweep)

Using the recursive splitter, we sweep `chunk_size` from 100 to 800 characters to observe how granularity influences retrieval precision.


In [ ]:
CHUNK_SIZES = [100, 200, 300, 500, 800]
OVERLAP     = 20

print(f"Chunk size sweep (recursive, overlap={OVERLAP}, top-{K}, {len(eval_pairs)} queries)\n")
print(f"{'chunk_size':>12}{'n_chunks':>10}{'Hit Rate':>12}")
print("-" * 34)

sweep_results = []

for size in CHUNK_SIZES:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.create_documents(
        texts=[d["text"] for d in raw_docs],
        metadatas=[{"title": d["title"]} for d in raw_docs],
    )
    col = f"chunk_sweep_{size}"
    store = build_collection(col, chunks)
    retriever = store.as_retriever(search_kwargs={"k": K})

    hits = []
    for pair in eval_pairs:
        docs = retriever.invoke(pair["question"])
        hits.append(retrieval_hit(docs, pair["answer"]))

    hit_rate = mean(hits) * 100
    sweep_results.append({"chunk_size": size, "n_chunks": len(chunks), "hit_rate": hit_rate})
    print(f"{size:>12}{len(chunks):>10}{hit_rate:>11.1f}%")

    # Clean up sweep collection
    client.delete_collection(col)


In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4))

chunk_sizes = [r["chunk_size"] for r in sweep_results]
hit_rates   = [r["hit_rate"] for r in sweep_results]
n_chunks    = [r["n_chunks"] for r in sweep_results]

color_hit = "#C44E52"
ax1.plot(chunk_sizes, hit_rates, marker="o", color=color_hit, linewidth=2.2, label="Hit rate (%)")
ax1.set_xlabel("chunk_size (characters)")
ax1.set_ylabel("Hit Rate (%)", color=color_hit)
ax1.tick_params(axis="y", labelcolor=color_hit)
ax1.grid(True, linestyle="--", alpha=0.5)

ax2 = ax1.twinx()
color_n = "#4C72B0"
ax2.bar(chunk_sizes, n_chunks, width=30, alpha=0.3, color=color_n, label="Total chunks")
ax2.set_ylabel("Total chunks produced", color=color_n)
ax2.tick_params(axis="y", labelcolor=color_n)

plt.title("Effect of Chunk Size on Retrieval Hit Rate (Recursive Splitter)")
fig.tight_layout()
plt.show()


In [ ]:
for strategy_name in stores:
    col = f"chunk_eval_{strategy_name}"
    if client.collection_exists(col):
        client.delete_collection(col)
        print(f"Deleted '{col}'")


---
## Summary & Strategy Trade-Offs

| Strategy | Splitting Mechanism | Pros | Cons |
|---|---|---|---|
| **Fixed-size** | Character count + delimiter | Extremely fast; predictable chunk lengths | May split facts or sentences in half |
| **Recursive** | Hierarchy of separators (`\n\n` $\to$ `\n` $\to$ `. ` $\to$ ` `) | Keeps paragraphs & sentences intact; general default | Heuristic-based, unaware of semantic shifts |
| **Sentence (NLTK)** | Punkt sentence tokenizer | Guarantees grammatical completeness | Variable chunk lengths on dialogue / short lines |
| **Semantic** | Cosine distance shift between sentence embeddings | Creates true topic-coherent semantic passages | Computationally heavy (sentence embedding inference) |

> **Key takeaway**: Recursive splitting with `chunk_size` around 300–500 characters and 10–20% overlap provides the best balance of context preservation and computational efficiency for general text RAG.

Next: `6.7-Retrieval-Hybrid-and-Fusion.ipynb` combines lexical and dense retrieval over the same passages.
